# 07 — File/rate streaming fallback (EMR, no MSK required)

Exercises the exact same streaming mechanics as `02_kafka_msk_streaming_ingest.ipynb` and `03_streaming_silver_gold_delta.ipynb`, without requiring a live MSK cluster. Use this while `infra/terraform` is being provisioned, or for a classroom/demo run with no AWS networking dependency.

This writes to `bronze_clickstream_rate`. Point `03`'s `bronze_table` variable at `bronze_clickstream_rate` to run the rest of the pipeline against this fallback source.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path)
spark.sql(f"USE `{cfg.schema}`")

In [ ]:
from pyspark.sql import functions as F

stream_events = (
    spark.readStream.format("rate")
    .option("rowsPerSecond", 50)
    .option("numPartitions", 2)
    .load()
    .select(
        F.concat(F.lit("rate-"), F.col("value")).alias("event_id"),
        F.col("timestamp").alias("event_ts"),
        F.concat(F.lit("u"), F.lpad((F.col("value") % 200).cast("string"), 5, "0")).alias("user_id"),
        F.concat(F.lit("s"), F.lpad((F.col("value") % 50).cast("string"), 5, "0")).alias("session_id"),
        F.concat(F.lit("p"), F.lpad(((F.col("value") % 50) + 1).cast("string"), 4, "0")).alias("product_id"),
        F.when((F.col("value") % 10) == 0, "purchase").when((F.col("value") % 5) == 0, "add_to_cart").otherwise("view").alias("event_type"),
        F.when((F.col("value") % 10) == 0, 1).otherwise(0).cast("int").alias("quantity"),
        F.when((F.col("value") % 10) == 0, 19.99).otherwise(0.0).cast("double").alias("price"),
        F.lit("product").alias("page"),
        F.lit("rate_source").alias("user_agent"),
    )
)

from retail_lakehouse.transformations import add_ingest_metadata
bronze_stream = add_ingest_metadata(stream_events, "rate_source_fallback")

query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", cfg.checkpoint("rate_bronze_clickstream"))
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(cfg.table("bronze_clickstream_rate"))
)

query.awaitTermination()
print("AvailableNow fallback stream completed")
spark.table(cfg.table("bronze_clickstream_rate")).orderBy(F.desc("event_ts")).limit(20).show(truncate=False)

## Next

Run `03_streaming_silver_gold_delta.ipynb` with its `bronze_table` variable set to `bronze_clickstream_rate` to build silver/gold from this fallback source using the same package code that will run against real MSK data.